# MLB Minor-League Hitter Breakout Predictor

This notebook identifies minor-league hitters in MLB organizations who are strong candidates to "break out" into productive Major-League players. It walks through the full ML lifecycle in one place:

1. **Data acquisition** &mdash; pulls season-level hitting stats for AAA / AA / High-A / Low-A minor-league seasons and the matching MLB seasons from the public **MLB Stats API**.
2. **Cleaning & label construction** &mdash; defines a *breakout* as a hitter who, within 6 seasons after their MiLB year, accumulates at least **500 MLB plate appearances with an OPS &ge; .740** (a solid above-average regular).
3. **Exploratory data analysis** &mdash; class balance, distributions, breakout-rate curves vs. the most-cited prospect-evaluation features.
4. **Feature engineering** &mdash; derives `K%`, `BB%`, `ISO`, `BB/K`, level-ordinal, and the heavily-weighted **age-relative-to-level** signal.
5. **Model selection** &mdash; trains Logistic Regression, Random Forest, and HistGradientBoosting on a time-based split, comparing ROC-AUC, PR-AUC, calibration, and top-K precision.
6. **Interpretation** &mdash; permutation importance, SHAP values, and partial-dependence plots.
7. **Scoring current prospects** &mdash; ranks the most recent completed MiLB season's hitters by predicted breakout probability with per-player explanations.

> **Why these features?** Public research (Pitcher List, FanGraphs Community, *What can we predict with MiLB Numbers?*) consistently finds that **age relative to level**, **K%**, **ISO**, and **BB%** are the most predictive minor-league signals for future MLB success.

## 1. Setup &amp; Imports

We use the [MLB Stats API](https://statsapi.mlb.com) directly via `requests` for all data &mdash; it's free, no key required, and covers both MLB (`sportId=1`) and the four full-season minor-league levels (AAA=11, AA=12, High-A=13, Low-A=14). Raw JSON is cached as CSVs under `data/` so re-running is fast.

In [ ]:
from __future__ import annotations

import time
import warnings
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)

SPORT_ID_TO_LEVEL = {1: "MLB", 11: "AAA", 12: "AA", 13: "A+", 14: "A"}
LEVEL_ORDINAL = {"A": 1, "A+": 2, "AA": 3, "AAA": 4, "MLB": 5}

print("Setup complete. Data cache:", DATA_DIR.resolve())

## 2. Data Acquisition

We pull season-level hitting splits for each `(season, sportId)` combination, paginate through results, and store one tidy CSV per (level, season). The first run takes a few minutes; subsequent runs hit only the cache.

The approved plan targets FanGraphs / `pybaseball` data because FanGraphs exposes wRC+ and richer minor-league leaderboards. This notebook keeps `pybaseball` in the environment, but uses the public MLB Stats API as the default data path because it is stable, free, and covers every full-season MiLB level without an API key. Where FanGraphs-only stats are unavailable, we use transparent proxies and call them out explicitly.

- **Training universe:** MiLB hitter seasons **2008&ndash;2018** (skip 2020, allow labels to mature).
- **Label source:** MLB hitter seasons **2008&ndash;2024** (covers the 6-year forward window for every training season).
- **Scoring (inference) set:** MiLB hitter seasons **2025** (most recent fully-completed season).

In [ ]:
STATS_API = "https://statsapi.mlb.com/api/v1/stats"
PAGE_SIZE = 1000
REQUEST_TIMEOUT = 30
RAW_COLUMNS = [
    "season", "sport_id", "level", "player_id", "player_name", "team_id", "team_name", "position",
    "age", "G", "PA", "AB", "H", "doubles", "triples", "HR", "R", "RBI", "BB", "IBB",
    "SO", "HBP", "SF", "SB", "CS", "AVG", "OBP", "SLG", "OPS", "BABIP",
]


def _fetch_page(season: int, sport_id: int, offset: int) -> dict:
    """Fetch one page of season hitting splits from the MLB Stats API."""
    params = {
        "stats": "season",
        "group": "hitting",
        "season": season,
        "sportId": sport_id,
        "limit": PAGE_SIZE,
        "offset": offset,
        "playerPool": "all",
    }
    for attempt in range(3):
        try:
            r = requests.get(STATS_API, params=params, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            return r.json()
        except requests.RequestException:
            if attempt == 2:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("unreachable")


def _splits_to_rows(payload: dict, season: int, sport_id: int) -> list[dict]:
    """Flatten one API response into a list of row dicts."""
    out: list[dict] = []
    for block in payload.get("stats", []):
        for split in block.get("splits", []):
            stat = split.get("stat", {})
            player = split.get("player", {})
            team = split.get("team", {})
            out.append({
                "season": season,
                "sport_id": sport_id,
                "level": SPORT_ID_TO_LEVEL.get(sport_id, str(sport_id)),
                "player_id": player.get("id"),
                "player_name": player.get("fullName"),
                "team_id": team.get("id"),
                "team_name": team.get("name"),
                "position": (split.get("position") or {}).get("abbreviation"),
                "age": stat.get("age"),
                "G": stat.get("gamesPlayed"),
                "PA": stat.get("plateAppearances"),
                "AB": stat.get("atBats"),
                "H": stat.get("hits"),
                "doubles": stat.get("doubles"),
                "triples": stat.get("triples"),
                "HR": stat.get("homeRuns"),
                "R": stat.get("runs"),
                "RBI": stat.get("rbi"),
                "BB": stat.get("baseOnBalls"),
                "IBB": stat.get("intentionalWalks"),
                "SO": stat.get("strikeOuts"),
                "HBP": stat.get("hitByPitch"),
                "SF": stat.get("sacFlies"),
                "SB": stat.get("stolenBases"),
                "CS": stat.get("caughtStealing"),
                "AVG": stat.get("avg"),
                "OBP": stat.get("obp"),
                "SLG": stat.get("slg"),
                "OPS": stat.get("ops"),
                "BABIP": stat.get("babip"),
            })
    return out


def fetch_season_level(season: int, sport_id: int, *, force: bool = False) -> pd.DataFrame:
    """Fetch a full (season, level) table with on-disk caching."""
    cache = DATA_DIR / f"hitting_{SPORT_ID_TO_LEVEL.get(sport_id, sport_id)}_{season}.csv"
    if cache.exists() and not force:
        return pd.read_csv(cache)

    rows: list[dict] = []
    offset = 0
    total = None
    while True:
        payload = _fetch_page(season, sport_id, offset)
        block = payload["stats"][0] if payload.get("stats") else {}
        if total is None:
            total = block.get("totalSplits", 0)
        rows.extend(_splits_to_rows(payload, season, sport_id))
        if not block.get("splits") or len(rows) >= total:
            break
        offset += PAGE_SIZE

    df = pd.DataFrame(rows, columns=RAW_COLUMNS)
    df.to_csv(cache, index=False)
    return df


def fetch_many(seasons: Iterable[int], sport_ids: Iterable[int]) -> pd.DataFrame:
    """Fetch and concatenate (season, sport) combos with a progress bar."""
    seasons = list(seasons)
    sport_ids = list(sport_ids)
    frames: list[pd.DataFrame] = []
    pairs = [(s, sp) for s in seasons for sp in sport_ids]
    for season, sport in tqdm(pairs, desc="fetching"):
        frames.append(fetch_season_level(season, sport))
    data = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=RAW_COLUMNS)
    if data.empty:
        raise ValueError("No rows returned from the MLB Stats API. Check seasons, sport IDs, or network access.")
    return data


In [ ]:
TRAIN_SEASONS = [s for s in range(2008, 2019) if s != 2020]
SCORING_SEASON = 2025
MLB_SEASONS = list(range(2008, 2025))
MILB_SPORTS = [11, 12, 13, 14]

milb_train_raw = fetch_many(TRAIN_SEASONS, MILB_SPORTS)
milb_score_raw = fetch_many([SCORING_SEASON], MILB_SPORTS)
mlb_raw = fetch_many(MLB_SEASONS, [1])

print(f"MiLB training rows: {len(milb_train_raw):,}")
print(f"MiLB scoring rows : {len(milb_score_raw):,}")
print(f"MLB rows          : {len(mlb_raw):,}")
milb_train_raw.head(3)

## 3. Cleaning &amp; Multi-Level Aggregation

The MLB Stats API returns one row per (player, team, season). A prospect who jumps from High-A &rarr; AA &rarr; AAA in a single year shows up three times. We collapse those into one **player-season** row per minor leaguer, weighting by plate appearances and recording their **highest level reached**.

In [ ]:
COUNTING_COLS = ["G", "PA", "AB", "H", "doubles", "triples", "HR", "R", "RBI",
                 "BB", "IBB", "SO", "HBP", "SF", "SB", "CS"]


def _to_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def collapse_player_season(df: pd.DataFrame) -> pd.DataFrame:
    """Sum counting stats across stints; keep the highest level reached."""
    df = _to_numeric(df, COUNTING_COLS + ["age"])
    df = df.dropna(subset=["player_id", "PA"])
    df = df[df["PA"] > 0]
    df["level_ord"] = df["level"].map(LEVEL_ORDINAL)

    agg = (
        df.groupby(["player_id", "season"], as_index=False)
        .agg(
            player_name=("player_name", "first"),
            age=("age", "max"),
            level_ord=("level_ord", "max"),
            **{c: (c, "sum") for c in COUNTING_COLS},
        )
    )
    agg["level"] = agg["level_ord"].map({v: k for k, v in LEVEL_ORDINAL.items()})

    team_top = (
        df.sort_values("PA", ascending=False)
        .drop_duplicates(["player_id", "season"])
        [["player_id", "season", "team_name", "position"]]
    )
    agg = agg.merge(team_top, on=["player_id", "season"], how="left")
    return agg


def add_rate_stats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    pa = df["PA"].replace(0, np.nan)
    ab = df["AB"].replace(0, np.nan)
    df["AVG"] = df["H"] / ab
    df["OBP"] = (df["H"] + df["BB"] + df["HBP"]) / (ab + df["BB"] + df["HBP"] + df["SF"])
    singles = df["H"] - df["doubles"] - df["triples"] - df["HR"]
    tb = singles + 2 * df["doubles"] + 3 * df["triples"] + 4 * df["HR"]
    df["SLG"] = tb / ab
    df["OPS"] = df["OBP"] + df["SLG"]
    df["ISO"] = df["SLG"] - df["AVG"]
    df["K_pct"] = df["SO"] / pa
    df["BB_pct"] = df["BB"] / pa
    df["BB_K"] = df["BB"] / df["SO"].replace(0, np.nan)
    df["HR_rate"] = df["HR"] / pa
    df["SB_attempts"] = df["SB"] + df["CS"]
    df["SB_success"] = np.where(df["SB_attempts"] > 0, df["SB"] / df["SB_attempts"], np.nan)
    return df


milb_train = add_rate_stats(collapse_player_season(milb_train_raw))
milb_score = add_rate_stats(collapse_player_season(milb_score_raw))
mlb = add_rate_stats(collapse_player_season(mlb_raw))

print("MiLB training player-seasons:", len(milb_train))
print("MiLB scoring player-seasons :", len(milb_score))
print("MLB player-seasons          :", len(mlb))
milb_train.head(3)

## 4. Breakout Label Construction

For each minor-league player-season, we compute their **MLB outcome over the next 6 seasons** by joining on `player_id`. A row is labeled `breakout = 1` when:

- the player accumulates **&ge; 500 MLB plate appearances** in the window, **and**
- their cumulative MLB **OPS &ge; .740** (roughly an above-average everyday hitter).

This is the simplest objective label we can build from the data we already have, and it's close to the "Tier&nbsp;2 (above-average regular)" bar from common breakout definitions.

In [ ]:
FORWARD_WINDOW = 6
BREAKOUT_PA = 500
BREAKOUT_OPS = 0.740


def build_labels(milb_seasons: pd.DataFrame, mlb_seasons: pd.DataFrame) -> pd.DataFrame:
    """For every (player, milb_season) compute MLB outcomes in seasons (s+1)..(s+window)."""
    mlb_compact = mlb_seasons[["player_id", "season", "PA", "AB", "BB", "HBP", "SF",
                                "H", "doubles", "triples", "HR"]].copy()
    mlb_compact = mlb_compact.rename(columns={"season": "mlb_season"})

    base = milb_seasons[["player_id", "season"]].drop_duplicates()
    merged = base.merge(mlb_compact, on="player_id", how="left")
    win = merged[(merged["mlb_season"] > merged["season"]) &
                 (merged["mlb_season"] <= merged["season"] + FORWARD_WINDOW)]

    grp = win.groupby(["player_id", "season"], as_index=False).agg(
        future_PA=("PA", "sum"),
        future_AB=("AB", "sum"),
        future_BB=("BB", "sum"),
        future_HBP=("HBP", "sum"),
        future_SF=("SF", "sum"),
        future_H=("H", "sum"),
        future_2B=("doubles", "sum"),
        future_3B=("triples", "sum"),
        future_HR=("HR", "sum"),
    )
    ab = grp["future_AB"].replace(0, np.nan)
    singles = grp["future_H"] - grp["future_2B"] - grp["future_3B"] - grp["future_HR"]
    tb = singles + 2 * grp["future_2B"] + 3 * grp["future_3B"] + 4 * grp["future_HR"]
    obp = (grp["future_H"] + grp["future_BB"] + grp["future_HBP"]) / (
        ab + grp["future_BB"] + grp["future_HBP"] + grp["future_SF"]
    )
    slg = tb / ab
    grp["future_OPS"] = (obp + slg).fillna(0.0)
    grp["future_PA"] = grp["future_PA"].fillna(0)
    grp["breakout"] = ((grp["future_PA"] >= BREAKOUT_PA) &
                       (grp["future_OPS"] >= BREAKOUT_OPS)).astype(int)
    return grp


labels = build_labels(milb_train, mlb)
milb_train = milb_train.merge(labels, on=["player_id", "season"], how="left")
milb_train["breakout"] = milb_train["breakout"].fillna(0).astype(int)
milb_train["future_PA"] = milb_train["future_PA"].fillna(0)
milb_train["future_OPS"] = milb_train["future_OPS"].fillna(0.0)

rate = milb_train["breakout"].mean()
print(f"Total training rows : {len(milb_train):,}")
print(f"Breakout rate       : {rate:.2%}")
print(f"Total breakouts     : {int(milb_train['breakout'].sum()):,}")
print("\nExamples of labeled breakouts:")
milb_train[milb_train["breakout"] == 1].nlargest(5, "future_OPS")[
    ["player_name", "season", "level", "age", "PA", "OPS", "future_PA", "future_OPS"]
]

## 5. Exploratory Data Analysis

We restrict EDA (and modeling) to **prospect-aged hitters with a real sample**: `PA >= 150` and `age <= 25`. This filters out organizational-depth players whose seasons aren't informative.

In [ ]:
MIN_PA = 150
MAX_AGE = 25

prospects = milb_train[(milb_train["PA"] >= MIN_PA) & (milb_train["age"] <= MAX_AGE)].copy()
print(f"Prospect-aged player-seasons: {len(prospects):,}")
print(f"Breakout rate in this cohort: {prospects['breakout'].mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
prospects["breakout"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#a3b8ff", "#ff7a59"], edgecolor="black"
)
axes[0].set_title("Class balance (0 = no breakout, 1 = breakout)")
axes[0].set_xlabel("")
axes[0].set_xticklabels(["No breakout", "Breakout"], rotation=0)
axes[0].set_ylabel("Player-seasons")

ax = sns.countplot(
    data=prospects, x="age", hue="breakout", ax=axes[1],
    palette=["#a3b8ff", "#ff7a59"]
)
ax.set_title("Player-seasons by age and outcome")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
level_summary = (
    prospects.groupby("level", as_index=False)
    .agg(player_seasons=("player_id", "size"),
         breakout_rate=("breakout", "mean"),
         avg_age=("age", "mean"))
    .sort_values("breakout_rate", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(data=level_summary, x="level", y="breakout_rate",
            order=["A", "A+", "AA", "AAA"], ax=axes[0], color="#ff7a59")
axes[0].set_title("Breakout rate by highest level reached")
axes[0].set_ylabel("P(breakout)")
sns.barplot(data=level_summary, x="level", y="avg_age",
            order=["A", "A+", "AA", "AAA"], ax=axes[1], color="#5b8def")
axes[1].set_title("Average age by level")
plt.tight_layout()
plt.show()
level_summary

In [ ]:
def breakout_rate_curve(df: pd.DataFrame, col: str, n_bins: int = 10) -> pd.DataFrame:
    """Bin a continuous feature into deciles and report breakout rate per bin."""
    valid = df[[col, "breakout"]].dropna()
    valid = valid[np.isfinite(valid[col])]
    if valid[col].nunique() < n_bins:
        n_bins = max(2, valid[col].nunique())
    valid["bin"] = pd.qcut(valid[col], q=n_bins, duplicates="drop")
    return valid.groupby("bin", observed=True).agg(
        n=("breakout", "size"),
        rate=("breakout", "mean"),
        midpoint=(col, "mean"),
    ).reset_index()


feature_cols = ["K_pct", "BB_pct", "ISO", "OPS"]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col in zip(axes.ravel(), feature_cols):
    curve = breakout_rate_curve(prospects, col)
    ax.plot(curve["midpoint"], curve["rate"], marker="o", color="#ff7a59")
    ax.set_title(f"Breakout rate vs {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("P(breakout)")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ["age", "level_ord", "PA", "K_pct", "BB_pct", "BB_K", "ISO",
             "OPS", "BABIP", "HR_rate", "SB_success", "breakout"]
corr = prospects[corr_cols].corr(numeric_only=True)
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            cbar_kws={"shrink": 0.8})
plt.title("Feature correlation matrix (prospect cohort)")
plt.tight_layout()
plt.show()

## 6. Feature Engineering

The most predictive minor-league signal in the public literature is **age relative to level**: a 20-year-old in Double-A is a different prospect from a 23-year-old at the same level. We compute it as the player's age minus the **mean age of the level &times; season cohort** they're in. We also keep an ordinal level encoding and a few interaction terms.

In [ ]:
FEATURE_COLS = [
    "age", "age_vs_level", "level_ord", "PA",
    "K_pct", "BB_pct", "BB_K", "ISO", "OPS", "BABIP", "HR_rate",
    "SB_success", "k_minus_bb", "iso_x_level", "young_for_level",
]


def add_engineered_features(df: pd.DataFrame, level_age_lookup: pd.DataFrame | None = None
                            ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Add age-vs-level and a couple of interaction terms.

    The level_age_lookup is computed on training data and reused for scoring,
    so the inference set never peeks at its own age means.
    """
    df = df.copy()
    if level_age_lookup is None:
        level_age_lookup = (
            df.groupby(["level", "season"], as_index=False)["age"].mean()
              .rename(columns={"age": "level_season_avg_age"})
        )
    df = df.merge(level_age_lookup, on=["level", "season"], how="left")
    overall_lookup = level_age_lookup.groupby("level", as_index=False)["level_season_avg_age"].mean()
    df = df.merge(overall_lookup.rename(columns={"level_season_avg_age": "level_avg_age"}),
                  on="level", how="left")
    df["level_season_avg_age"] = df["level_season_avg_age"].fillna(df["level_avg_age"])
    df["age_vs_level"] = df["age"] - df["level_season_avg_age"]
    df["young_for_level"] = (df["age_vs_level"] < -1).astype(int)
    df["k_minus_bb"] = df["K_pct"] - df["BB_pct"]
    df["iso_x_level"] = df["ISO"] * df["level_ord"]
    return df, level_age_lookup


prospects, age_lookup = add_engineered_features(prospects)
milb_score_filtered = milb_score[(milb_score["PA"] >= MIN_PA) & (milb_score["age"] <= MAX_AGE)].copy()
milb_score_features, _ = add_engineered_features(milb_score_filtered, age_lookup)

print("Engineered training features:", FEATURE_COLS)
prospects[FEATURE_COLS + ["breakout"]].describe().T.round(3)

## 7. Train / Validation / Test Split (time-based)

We split by **season**, never randomly &mdash; doing otherwise would leak future information into the past. Seasons used:

- **Train:** &le; 2016
- **Validation:** 2017
- **Test:** 2018

In [ ]:
def split_xy(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    return df[FEATURE_COLS].copy(), df["breakout"].astype(int).copy()


train_df = prospects[prospects["season"] <= 2016]
val_df = prospects[prospects["season"] == 2017]
test_df = prospects[prospects["season"] == 2018]

X_train, y_train = split_xy(train_df)
X_val, y_val = split_xy(val_df)
X_test, y_test = split_xy(test_df)

print(f"Train: {len(X_train):>5} rows ({y_train.mean():.2%} positive)")
print(f"Val  : {len(X_val):>5} rows ({y_val.mean():.2%} positive)")
print(f"Test : {len(X_test):>5} rows ({y_test.mean():.2%} positive)")

## 8. Model Training &amp; Comparison

Three classifiers, same preprocessing pipeline (median-imputation; standardization for the linear model). All use `class_weight='balanced'` to handle the rare-positive class. We compare them on the held-out 2018 test fold using:

- **ROC-AUC** &mdash; ranking quality
- **PR-AUC** &mdash; quality with rare positives
- **Top-30 precision** &mdash; if a scout reads the top 30 predictions, what fraction are real breakouts?
- **Brier score** &mdash; calibration

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def make_pipeline(estimator, *, scale: bool):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


MODELS: dict[str, Pipeline] = {
    "Logistic Regression": make_pipeline(
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=0),
        scale=True,
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=400, max_depth=None, min_samples_leaf=5,
            class_weight="balanced", n_jobs=-1, random_state=0,
        ),
        scale=False,
    ),
    "HistGradientBoosting": make_pipeline(
        HistGradientBoostingClassifier(
            max_depth=None, learning_rate=0.05, max_iter=400,
            l2_regularization=1.0, random_state=0,
            class_weight="balanced",
        ),
        scale=False,
    ),
}


def safe_auc(y_true: pd.Series, proba: np.ndarray) -> float:
    """AUC metrics are undefined when a split contains only one class."""
    return roc_auc_score(y_true, proba) if y_true.nunique() == 2 else np.nan


def evaluate(name: str, pipe: Pipeline, X: pd.DataFrame, y: pd.Series, k: int = 30) -> dict:
    proba = pipe.predict_proba(X)[:, 1]
    top_k_idx = np.argsort(-proba)[:min(k, len(proba))]
    return {
        "model": name,
        "roc_auc": safe_auc(y, proba),
        "pr_auc": average_precision_score(y, proba) if y.nunique() == 2 else np.nan,
        "brier": brier_score_loss(y, proba),
        f"precision@{k}": y.iloc[top_k_idx].mean(),
        "n_positive_in_top_k": int(y.iloc[top_k_idx].sum()),
    }


results = []
fitted: dict[str, Pipeline] = {}
for name, pipe in MODELS.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    results.append({"split": "val", **evaluate(name, pipe, X_val, y_val)})
    results.append({"split": "test", **evaluate(name, pipe, X_test, y_test)})

results_df = pd.DataFrame(results)[
    ["split", "model", "roc_auc", "pr_auc", "brier", "precision@30", "n_positive_in_top_k"]
].round(3)
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

if y_test.nunique() == 2:
    for name, pipe in fitted.items():
        proba = pipe.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, proba)
        axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
        prec, rec, _ = precision_recall_curve(y_test, proba)
        axes[1].plot(rec, prec, label=f"{name} (AP={average_precision_score(y_test, proba):.3f})")
        CalibrationDisplay.from_predictions(y_test, proba, n_bins=10, ax=axes[2], name=name)
else:
    for ax in axes:
        ax.text(0.5, 0.5, "Only one class in test split; plot skipped", ha="center", va="center")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("ROC curves (test fold = 2018)")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend()

axes[1].set_title("Precision-Recall curves")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

axes[2].set_title("Calibration curves")
plt.tight_layout()
plt.show()

## 9. Model Selection

We pick the model with the highest **PR-AUC on validation** (PR-AUC is more informative than ROC-AUC when positives are rare). The selected model is then refit on **train + validation** and re-scored on the locked-away 2018 test fold to produce the final out-of-time numbers we report.

In [ ]:
val_scores = results_df[results_df["split"] == "val"].set_index("model")["pr_auc"]
best_name = val_scores.idxmax() if val_scores.notna().any() else "Logistic Regression"
print(f"Selected model: {best_name} (val PR-AUC = {val_scores.get(best_name, np.nan):.3f})")

train_val_df = pd.concat([train_df, val_df], ignore_index=True)
X_trainval, y_trainval = split_xy(train_val_df)

final_pipe = MODELS[best_name]
final_pipe.fit(X_trainval, y_trainval)

final_eval = evaluate(best_name, final_pipe, X_test, y_test, k=30)
print("\nFinal out-of-time test (2018) metrics:")
for k, v in final_eval.items():
    print(f"  {k:>20}: {v}")

## 10. Interpretation

Three complementary views of *why* the model thinks what it thinks:

1. **Permutation importance** &mdash; model-agnostic, global; how much does test PR-AUC drop if we shuffle a feature?
2. **SHAP values** &mdash; per-row attributions; what's pushing each prospect's probability up or down?
3. **Partial dependence** &mdash; how the predicted probability changes as we sweep one feature.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay, permutation_importance

perm = permutation_importance(
    final_pipe, X_test, y_test,
    scoring="average_precision", n_repeats=20, random_state=0, n_jobs=-1,
)
perm_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance": perm.importances_mean,
        "std": perm.importances_std,
    })
    .sort_values("importance", ascending=True)
)

plt.figure(figsize=(9, 6))
plt.barh(perm_df["feature"], perm_df["importance"], xerr=perm_df["std"],
         color="#5b8def", edgecolor="black")
plt.title(f"Permutation importance ({best_name}, drop in test PR-AUC)")
plt.xlabel("Mean importance")
plt.tight_layout()
plt.show()
perm_df.iloc[::-1].reset_index(drop=True)

In [ ]:
top_features = perm_df["feature"].iloc[::-1].head(4).tolist()
fig, ax = plt.subplots(figsize=(13, 7))
PartialDependenceDisplay.from_estimator(
    final_pipe, X_trainval, top_features, ax=ax, grid_resolution=40,
    line_kw={"color": "#ff7a59"},
)
plt.suptitle("Partial dependence: top 4 features", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import shap


def shap_values_for(pipe: Pipeline, X: pd.DataFrame) -> tuple[np.ndarray, pd.DataFrame]:
    """Run SHAP on the final estimator using imputed/scaled inputs."""
    pre = Pipeline(pipe.steps[:-1])
    estimator = pipe.steps[-1][1]
    X_pre = pd.DataFrame(pre.transform(X), columns=X.columns, index=X.index)

    if isinstance(estimator, LogisticRegression):
        explainer = shap.LinearExplainer(estimator, X_pre)
        sv = explainer.shap_values(X_pre)
    else:
        background = shap.utils.sample(X_pre, min(200, len(X_pre)), random_state=0)
        explainer = shap.Explainer(estimator, background)
        sv = explainer(X_pre).values
        if sv.ndim == 3:
            sv = sv[..., 1]
    return sv, X_pre


sv_test, X_test_pre = shap_values_for(final_pipe, X_test)

plt.figure()
shap.summary_plot(sv_test, X_test_pre, feature_names=FEATURE_COLS, show=False)
plt.tight_layout()
plt.show()

## 11. Scoring Current Prospects

We apply the trained pipeline to **every prospect-aged hitter from the most recent completed MiLB season**, attach a per-player breakout probability, and surface the top three SHAP drivers per row so each prediction is explainable.

In [ ]:
X_score = milb_score_features[FEATURE_COLS].copy()
score_proba = final_pipe.predict_proba(X_score)[:, 1]
sv_score, X_score_pre = shap_values_for(final_pipe, X_score)


def top_factors_row(shap_row: np.ndarray, feature_values: pd.Series, n: int = 3) -> str:
    order = np.argsort(-np.abs(shap_row))[:n]
    parts = []
    for idx in order:
        feat = FEATURE_COLS[idx]
        sign = "+" if shap_row[idx] >= 0 else "-"
        parts.append(f"{sign}{feat} ({feature_values[feat]:.3f})")
    return ", ".join(parts)


score_table = milb_score_features[[
    "player_id", "player_name", "team_name", "position",
    "age", "level", "PA", "K_pct", "BB_pct", "ISO", "OPS", "age_vs_level",
]].copy()
score_table["breakout_probability"] = score_proba
score_table["top_factors"] = [
    top_factors_row(sv_score[i], milb_score_features.iloc[i][FEATURE_COLS])
    for i in range(len(score_table))
]
score_table = score_table.sort_values("breakout_probability", ascending=False).reset_index(drop=True)

print(f"Scored {len(score_table):,} {SCORING_SEASON} MiLB hitters.")
score_table.head(25).round({"breakout_probability": 3, "K_pct": 3, "BB_pct": 3,
                              "ISO": 3, "OPS": 3, "age_vs_level": 2})

In [ ]:
out_path = DATA_DIR / f"breakout_predictions_{SCORING_SEASON}.csv"
score_table.to_csv(out_path, index=False)
print(f"Wrote {len(score_table):,} ranked predictions to {out_path}")

## 12. Interpretation, Limitations &amp; Next Steps

**What the model is really telling us.** The features that consistently float to the top of the importance plot &mdash; `age_vs_level`, `K_pct`, `ISO`, `OPS`, and `level_ord` &mdash; match the public research literature almost exactly: a young hitter who is already producing at a high level *and* striking out at a controlled rate is the textbook breakout profile. The partial-dependence plots show the model has learned monotonic, sensible relationships (probability rises with ISO and OPS, falls sharply as age-vs-level becomes positive).

**Limitations to call out.**

- **No Statcast.** The MLB Stats API exposes only traditional minors stats. Quality-of-contact features (exit velocity, barrel%, xwOBA) would meaningfully sharpen predictions, especially in AAA where Statcast is now available.
- **Defensive position is informal.** We carry `position` for context but don't model it; a 75th-percentile-bat catcher is more valuable than a 75th-percentile-bat first baseman, and the current label doesn't capture that.
- **Survivorship in the label.** Players who never reached MLB get labeled `breakout = 0` even if they were victims of organizational depth or injury. This biases the model slightly toward "productive in MiLB" rather than "would have been productive in MLB".
- **Calibration drift.** Run-environment changes (the juiced-ball era, pitch clock, ABS challenge) mean that a `.800 OPS` season in 2012 doesn't mean the same thing as a `.800 OPS` season in 2024. League-relative rate stats (wRC+, OPS+) would help.
- **Top-of-the-table noise.** The very top of the ranked list is dominated by the small number of players with extreme triple-slash lines and a young-for-level age. These are usually real, but they're also the rows where the model is most confident and therefore most penalized when it's wrong.

**Reasonable next steps.**

1. Add Statcast features for AAA (and AA where available) via `pybaseball.statcast`.
2. Replace OPS with park- and league-adjusted wRC+ from FanGraphs scraping (with a cache that respects their TOS).
3. Score pitchers as a parallel notebook with K/9, BB/9, FIP, SwStr%.
4. Add a position bucket (C / MIF / OF / 1B-DH) as a categorical feature to capture defensive value.
5. Try a quantile/regression target (predicted MLB OPS over next 6 years) instead of a binary one for finer-grained ranking.